# Inshallah Project: Phase 1 and Phase 2 notebook

This notebook installs and tests CAMeL Tools, downloads the three sources and the Tanzil text, finds every occurrence of إن شاء الله, builds D2.1 and selects the 1,000-row sample D2.2.

**Run:** upload this notebook to Colab, then Runtime → Run all. Kaggle keys go in Colab Secrets. The reviewed sheets are read from the `manual/` folder (see the README).

**Output:** the notebook writes D2.1, D2.2, the report tables and `inshallah_outputs.zip`. The corpus steps take a long time, so the project folder is kept in Google Drive. If the sample does not reach the expected minimum for every label, it is saved as provisional, not as D2.2.

**If the status says NEEDS_REVIEW:** fill the listed sheets, save them with the `_filled.csv` names and rerun from Step 4. Labels are never added just to meet a quota.


### Notes
Natural function estimates use the independent random check, never the enriched D2.2. RQ4 uses variant, position, verb_cue, future_marker and second_person. OTHER covers recognizable excluded functions; UNCLEAR means insufficient context. The inventory contains the three required boundary rules. Prior user-reviewed sheets are reused only where the occurrence and displayed context match. New or changed examples still need review. The 4–80 word requirement remains in effect, so bare formulas cannot be retained.

---
# Part A — Setup
## A1. Python version and runtime

In [ ]:
import sys, platform, os
print("Python", sys.version.split()[0], "|", platform.platform())
assert sys.version_info >= (3, 11), "CAMeL Tools 1.6 needs Python 3.11 or newer (Colab currently ships 3.12)."
IN_COLAB = "google.colab" in sys.modules
print("Running in Google Colab:", IN_COLAB)

## A2. Install packages
CAMeL Tools brings its own requirements (numpy 2, scikit-learn, torch, transformers; Colab already has torch and transformers).
If Colab shows a **"Restart session"** button after this cell, click it once and then run all cells again; the install is skipped the second time.

In [ ]:
SKIP_PIP = os.environ.get("INSHALLAH_SKIP_PIP") == "1"
if not SKIP_PIP:
    %pip install -q "camel-tools==1.6.0" "datasketch>=1.6" "huggingface_hub>=0.24" "kagglehub>=0.3" pyarrow "scipy>=1.11"

In [ ]:
import importlib.metadata as md
for pkg in ["camel-tools", "numpy", "pandas", "datasketch", "huggingface_hub", "pyarrow"]:
    try:
        print(f"{pkg:16s} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:16s} NOT INSTALLED")

## A3. Install CAMeL Tools data and test that it works
We download two morphology databases from the official CAMeL Tools catalogue:
**calima-glf-01** (Gulf Arabic, CC BY 4.0) and **calima-msa-r13** (MSA, used as a fallback).
The tests below must all pass; if one fails, the cell stops with an error.

In [ ]:
import subprocess
from pathlib import Path

def camel_db_installed(name):
    data_dir = Path(os.environ.get("CAMELTOOLS_DATA", Path.home() / ".camel_tools"))
    data_dir.mkdir(parents=True, exist_ok=True)
    return (data_dir / "data" / "morphology_db" / name / "morphology.db").exists()

for pkg, folder in [("morphology-db-glf-01", "calima-glf-01"), ("morphology-db-msa-r13", "calima-msa-r13")]:
    if not camel_db_installed(folder):
        print(f"Installing {pkg} ...")
        subprocess.run(["camel_data", "-i", pkg], check=True)
    else:
        print(f"{pkg} already installed")

In [ ]:
from camel_tools.utils.dediac import dediac_ar
from camel_tools.utils.normalize import (normalize_alef_ar, normalize_alef_maksura_ar,
                                         normalize_teh_marbuta_ar)
from camel_tools.tokenizers.word import simple_word_tokenize
from camel_tools.morphology.database import MorphologyDB
from camel_tools.morphology.analyzer import Analyzer

# Test 1: diacritic removal
assert dediac_ar("إِنْ شَاءَ اللَّهُ") == "إن شاء الله"
# Test 2: alef normalization
assert normalize_alef_ar("إن شاء الله") == "ان شاء الله"
# Test 3: tokenization separates punctuation
assert simple_word_tokenize("ان شاء الله، بروح بكرة!") == ["ان", "شاء", "الله", "،", "بروح", "بكرة", "!"]
# Test 4: morphological analysis with the Gulf database
GLF = Analyzer(MorphologyDB.builtin_db("calima-glf-01"))
MSA = Analyzer(MorphologyDB.builtin_db("calima-msa-r13"))
an = GLF.analyze("رحت")
assert any(a["pos"] == "verb" and a["asp"] == "p" and a["per"] == "1" for a in an), "Gulf analyzer should read رحت as 1st person past"
an = GLF.analyze("بنروح")
assert any(a["pos"] == "verb" and a.get("prc1") == "b_fut" for a in an), "Gulf analyzer should see the future prefix in بنروح"
assert len(MSA.analyze("كتبت")) > 0
print("CAMeL Tools: all 6 tests passed ✔")
print("Example — analyses of رحت:", sorted({(a['pos'], a['asp'], a['per']) for a in GLF.analyze('رحت')}))

## A4. Settings
Everything you may want to change is in this one cell. Each setting is explained next to it.

In [ ]:
import json, re, random, hashlib, datetime as dt, unicodedata, html
from collections import Counter, defaultdict
import numpy as np
import pandas as pd

SEED = 2026                     # fixed seed: same inputs -> same D2.2
random.seed(SEED); np.random.seed(SEED)

# Where everything is written. On Colab the default folder disappears when the session ends;
# set USE_DRIVE = True to keep your work (especially the filled manual sheets) in Google Drive.
USE_DRIVE = True  # preserves downloaded corpora and manual reviews across Colab sessions
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/inshallah_project")
else:
    PROJECT_DIR = Path(os.environ.get("INSHALLAH_PROJECT_DIR", "/content/inshallah_project" if IN_COLAB else "./inshallah_project")).resolve()

RAW = PROJECT_DIR / "data" / "raw"
INTERIM = PROJECT_DIR / "data" / "interim"
PROCESSED = PROJECT_DIR / "data" / "processed"
MANUAL = PROJECT_DIR / "manual"
REP1 = PROJECT_DIR / "reports" / "phase1"
REP2 = PROJECT_DIR / "reports" / "phase2"
for d in [RAW, INTERIM, PROCESSED, MANUAL, REP1, REP2]:
    d.mkdir(parents=True, exist_ok=True)
print("Project folder:", PROJECT_DIR)

# --- Sources ---------------------------------------------------------------
SFC_REPO = "faisalq/SFC-mini"
GULF_REPO = "AhmedSSabir/Gulf-Arabic-Tweets-2018-2020"
SADA_KAGGLE = "sdaiancai/sada2022"
SADA_HF_REPO = "MohamedRashad/SADA22"
SADA_SOURCE = "kaggle"          # "kaggle" (has file name + timestamps -> true previous segment) or "hf" (text only)
SFC_MAX_FILES = None            # None = all 63 files (~3 GB). Use e.g. 5 for a quick trial run.
TANZIL_URL = "https://tanzil.net/pub/download/index.php?quranType=simple-clean&outType=txt-2&agree=true"

# --- Cleaning ----------------------------------------------------------------
MIN_WORDS, MAX_WORDS = 4, 80    # host sentence length, from the Phase 2 guide
NEAR_DUP_THRESHOLD = 0.90       # Jaccard similarity above which two records are near-duplicates

# --- Quote detection -------------------------------------------------------
QUOTE_K = 4                     # match the Quran on 4-word windows (3 chained too many common phrases)
QUOTE_MIN_EXTRA = 2             # the matched run must contain >= 2 Quranic words besides the formula itself
FIXED_PRAYERS = [               # extend this list from what you see in the data
    "ما شاء الله تبارك الله",
    "لا حول ولا قوة الا بالله",
    "حسبنا الله ونعم الوكيل",
    "حسبي الله ونعم الوكيل",
    "انا لله وانا اليه راجعون",
    "بسم الله الرحمن الرحيم",
    "سبحان الله وبحمده سبحان الله العظيم",
]

# --- Labels ------------------------------------------------------------------
SIX_LABELS = ["COMMIT", "WISH", "DEFER", "HEDGE", "SKEPTIC", "CLOSE"]
ALL_QUICK_LABELS = SIX_LABELS + ["OTHER", "UNCLEAR"]   # OTHER added after professor feedback (item 7)

# --- Sampling (Step 7) -------------------------------------------------------
SOURCE_TOTALS = {"sfc": 300, "gulf": 350, "sada": 350}
# Strata are checked in this order; a row goes to the FIRST stratum it matches.
STRATA_ORDER = ["skeptic_cue", "short_or_alone", "answers_question", "addressed", "past_or_yakoon", "other"]
# Source-by-stratum quotas are solved in Step 7 from reviewed evidence.
EXTRA_PER_STRATUM = 20          # extra quick labels per targeted stratum (Step 6b); 0 to skip
RARE_CHECK_SIZES = {"sfc": 30, "gulf": 35, "sada": 35}   # the 100-row check mirrors D2.2's 300/350/350 mix
# Full D2.1 may be large. Optional caps are trial mode and are disclosed in the run manifest.
# Full pre-cap spelling counts are always exported separately.
D21_MAX_ROWS = {}  # full cleaned master by default; optionally {"sfc": 20000} for a trial

D21_COLUMNS = ["id", "source", "original_text", "host_sentence", "prev_context", "variant",
               "position", "verb_cue", "future_marker", "second_person", "quote_flag"]
D22_COLUMNS = D21_COLUMNS + ["stratum", "label", "context_needed"]

CHECK_FIELDS = ["variant", "position", "verb_cue", "future_marker", "second_person", "quote_flag"]
MIN_EXPECTED = 50
PREFERRED_EXPECTED = 60
MIN_RATE_REVIEWS = 10
RATE_SURVEY_PER_CELL = 20
RARE_CANDIDATES_PER_SOURCE = 100
REVIEWER_NAME = "Project team"
REVISION = "2"
RUN_STATUS = "NEEDS_REVIEW"

In [ ]:
# Restore your existing reviewed sheets only when a newer uploaded file is absent.
import base64, zlib, io
EMBEDDED_REVIEWS = json.loads(zlib.decompress(base64.b64decode('...')).decode("utf-8"))
for name, content in EMBEDDED_REVIEWS.items():
    target = MANUAL / name
    if not target.exists():
        target.write_text(content, encoding="utf-8-sig")
print("Prior validation restored; uploaded filled sheets take precedence.")


---
# Part B — Library
All the rules live here, each with a short test. Part C and Part D only call these functions.

## B1. Normalization that remembers positions
We search a **normalized copy** (no diacritics, no tatweel, one alef form), but the `variant`
column must hold the spelling **exactly as in `original_text`**. So the normalizer also returns
a map from every normalized character back to its position in the original text.

In [ ]:
DIACRITICS = re.compile(r"[ؐ-ًؚ-ٰٟۖ-ۭ]")
TATWEEL = "ـ"
ALEF_FORMS = set("آأإٱ")      # آ أ إ ٱ  -> ا

def normalize_with_map(text):
    """Return (normalized_text, idx) where idx[k] is the position in `text` of normalized char k."""
    out, idx = [], []
    for i, ch in enumerate(text):
        if DIACRITICS.match(ch) or ch == TATWEEL:
            continue
        out.append("ا" if ch in ALEF_FORMS else ch)
        idx.append(i)
    return "".join(out), idx

def norm(text):
    """Full search/dedupe normalization, done with CAMeL Tools (diacritics, alef, alef maksura, teh marbuta, tatweel)."""
    if not isinstance(text, str):
        return ""
    t = dediac_ar(text).replace(TATWEEL, "")
    t = normalize_alef_ar(t)
    t = normalize_alef_maksura_ar(t)
    t = normalize_teh_marbuta_ar(t)
    return t

# Our position-mapped normalizer must agree with CAMeL Tools on diacritics + alef + tatweel.
for s in ["إِنْ شَاءَ اللَّهُ", "إنـــشاء الله", "انْ شَاءَ اللّٰه", "أنا رحت", "ٱلله"]:
    mine, _ = normalize_with_map(s)
    camel = normalize_alef_ar(dediac_ar(s).replace(TATWEEL, ""))
    assert mine == camel, (s, mine, camel)
print("Normalizer agrees with CAMeL Tools ✔")

## B2. The formula finder
One pattern on the normalized text covers the spellings named in the brief (انشالله، ان شاء الله، إن شاء الله، إنشاءالله)
and other common ones (ان شالله، انشاء الله، ان شاء اللة). Word boundaries stop false hits such as *كان شاء* and *ما شاء الله*.
An attached و or ف is allowed before the formula but is **not** part of the recorded variant.

In [ ]:
AR_LETTER = r"ء-يٱ-ۓ"
FORMULA = re.compile(
    rf"(?:(?<![{AR_LETTER}])|(?<=(?<![{AR_LETTER}])[وف]))"   # start of word, or after a single attached و/ف
    r"ان\s*شا+\s*ء?\s*ا?لل[هة]+"                              # ان + شاء + الله, spaces optional
    rf"(?![{AR_LETTER}])"                                     # end of word
)

def find_formula(original):
    """Yield (start, end, variant) for each occurrence, with offsets into the ORIGINAL text."""
    if not isinstance(original, str) or not original:
        return
    normed, idx = normalize_with_map(original)
    for m in FORMULA.finditer(normed):
        start, end = idx[m.start()], idx[m.end() - 1] + 1
        # keep any diacritics that sit on the last letter
        while end < len(original) and (DIACRITICS.match(original[end]) or original[end] == TATWEEL):
            end += 1
        yield start, end, original[start:end]

def _variants(s):
    return [v for _, _, v in find_formula(s)]

_tests = {
    "انشالله بجي": ["انشالله"],
    "ان شاء الله": ["ان شاء الله"],
    "إن شاء الله": ["إن شاء الله"],
    "إنشاءالله": ["إنشاءالله"],
    "إِنْ شَاءَ اللَّهُ تعالى": ["إِنْ شَاءَ اللَّهُ"],
    "ان شالله": ["ان شالله"],
    "انشاء الله": ["انشاء الله"],
    "ان شاء اللة": ["ان شاء اللة"],
    "وان شاء الله نلتقي": ["ان شاء الله"],
    "إن شـــاء الله": ["إن شـــاء الله"],
    "ما شاء الله تبارك الله": [],
    "لو كان شاء الله": [],
    "ان شاء الله وان شاء الله": ["ان شاء الله", "ان شاء الله"],
}
for s, expected in _tests.items():
    assert _variants(s) == expected, (s, _variants(s), expected)
print(f"Formula finder: {len(_tests)} spelling tests passed ✔")

## B3. Sentences, previous sentence, and the working copy
* **host_sentence**: split on `. ! ? ؟ …` and line breaks. Tweets often have no punctuation, so the whole tweet is then one sentence.
* **prev_context**: the sentence before it in the **same** record. For SADA, the previous transcript segment of the same recording is used, as required by the guide.
* **Links and user names are removed from the working copies** (`host_sentence`, `prev_context`), never from `original_text`.

In [ ]:
SENT_END = re.compile(r"[.!?؟…\n\r]+")
URL_RE = re.compile(r"(https?://\S*|www\.\S*|pic\.twitter\.com/\S*)", re.I)
MENTION_RE = re.compile(r"@\w+:?")
RT_RE = re.compile(r"^\s*RT\b:?", re.I)
WORD_RE = re.compile(rf"[{AR_LETTER}A-Za-z0-9]")

def clean_working(s):
    if not isinstance(s, str):
        return ""
    s = html.unescape(s)
    s = URL_RE.sub(lambda m: " " + m.group()[len(m.group().rstrip(".!?؟…،")):], s)
    s = RT_RE.sub(" ", s)
    s = MENTION_RE.sub(" ", s)
    return re.sub(r"\s+", " ", s).strip()

def sentence_spans(text):
    """List of (start, end) spans of sentences in the original text."""
    spans, start = [], 0
    url_spans = [(m.start(), m.start() + len(m.group().rstrip(".!?؟…،"))) for m in URL_RE.finditer(text)]
    for m in SENT_END.finditer(text):
        if any(a <= m.start() < b for a, b in url_spans):
            continue
        if m.start() > start:
            spans.append((start, m.end()))
        start = m.end()
    if start < len(text):
        spans.append((start, len(text)))
    out = []
    for a, b in spans:
        while a < b and text[a].isspace():
            a += 1
        if a < b:
            out.append((a, b))
    return out

def words(s):
    """Word tokens (CAMeL tokenizer, punctuation and emoji dropped)."""
    return [t for t in simple_word_tokenize(s) if WORD_RE.search(t)]

def n_words(s):
    return len(words(s))

def is_hashtag_only(record):
    toks = clean_working(record).split()
    return len(toks) == 0 or all(t.startswith("#") for t in toks)

assert sentence_spans("متى بتجي؟ ان شاء الله. بكرة") == [(0, 9), (10, 22), (23, 27)]
assert clean_working("RT @ali_99: ان شاء الله https://t.co/x") == "ان شاء الله"
assert n_words("ان شاء الله، بروح بكرة! 😊") == 5
assert is_hashtag_only("#السعودية #الرياض https://t.co/a") and not is_hashtag_only("#الرياض ان شاء الله")
print("Sentence and cleaning tests passed ✔")

## B4. The descriptive columns (Step 4 of the Phase 2 guide)
The guide's rules, with these documented refinements (all of them are checked in the 50-row check):

* **position** — `first` / `middle` / `last` / `alone`, counted over word tokens of the host sentence (punctuation and emoji ignored).
* **verb_cue** — looks at the 3 words before and after the formula. Priority when several match: `me_future` > `me_past` > `other` > `none`.
  * `me_future`: first-person future/present verb — ب + verb without a 2nd/3rd-person prefix (بروح، بسوي، بنروح), راح/رح + verb starting with ا or ن, or a first-person imperfective verb according to the CAMeL Gulf analyzer.
  * `me_past`: word ending in ت or نا that the analyzer reads as a first-person past verb (رحت، قلت، سوينا).
  * `other`: any other word that the analyzer mostly reads as a verb.
* **future_marker** — the guide's rule (راح, or ب + at least 3 letters), refined: words starting with بال, a stop-list of common non-verbs (بكرة، بعد، بس، بيت…), and words that CAMeL cannot read as a verb with the future ب are not counted. راح followed by a noun with ال ("went to") is not counted.
* **second_person** — the guide's list (عندك، لكم، لك، انتم، انتو، انتي، انت، عليك) plus similar forms, the endings كم / كي / ك with an exception list (بارك، مبارك، ذلك، هناك…), and a vocative يا + person (not يا رب / يا الله).

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=100000)
def analyses(word):
    w = norm(word)
    a = GLF.analyze(w)
    if not a:
        a = MSA.analyze(w)
    return tuple((x.get("pos"), x.get("asp"), x.get("per"), x.get("prc1"), x.get("enc0")) for x in a)

def strip_conj(w):
    return w[1:] if len(w) > 3 and w[0] in "وف" else w

def forms(w):
    """The normalized word, plus the word without an attached و/ف (فيكم -> فيكم and يكم)."""
    t = norm(w)
    return [t] + ([t[1:]] if len(t) > 3 and t[0] in "وف" else [])

BA_STOP = set(norm(w) for w in """بكرة بكره بعد بعدين بعض بين بيت بيوت بنت بنات بلد بلاد بس بسم بسبب بدون برضو برضه بشر بحر بنك
بطل بكل بكلمة بحق بحب بحياة بخير بشي بشيء بصراحه بصراحة بصدق بسرعة بسرعه بإذن باذن بالله بإمكاني بامكاني بإمكانك بامكانك بدال
بداية بدايه برنامج بريد بعيد بقية بقيه بكم بكثير بشكل بوقت بنفس بجد بجنون بحث بيان بيع بطريقة بطريقه بطاقة بطاقه
بارك باركه بارك الله باري باب بابا بلاش بالضبط برا بره برع بلا""".split())
NOT_VERB = set(norm(w) for w in "يا رب ربي الله والله بس ما لا لم لن لو اذا اللي الي على في من عن مع هو هي انا احنا انت خير".split())

FUT_WORDS = {"راح", "رح"}
SECOND_PERSON_WORDS = set(norm(w) for w in """انت انتي انتِ انتم انتو انتوا انتن عندك عندكم عندكي لك لكم لكي عليك عليكم عليكي
منك منكم فيك فيكم معك معكم بك اياك اياكم الك الكم يعطيك يعطيكم يحفظك""".split())
K_EXCEPT = set(norm(w) for w in """ذلك كذلك لذلك بذلك فذلك تلك هناك هنالك بارك مبارك تبارك المبارك ملك الملك شك سمك ضحك ترك شارك
مشارك اشترك مشترك بنك البنك شباك هلاك ادراك اشراك سلك مسلك ديك هيك ذاك هذاك كذاك مسك فلك الفلك تبوك مبروك ممسك مملكه مملكة
المملكه المملكة ملوك سلوك شوك مالك عالك حالك""".split())
KUM_KI_EXCEPT = set(norm(w) for w in "حكم الحكم حاكم محكم بكم كم كي لكي ذكي تركي امريكي ملكي مكي زكي بكي شكي".split())
VOCATIVE_GOD = set(norm(w) for w in "رب ربي الله رحمن كريم".split())
QUESTION_WORDS = set(norm(w) for w in "متى وش ايش ليش ليه كيف هل ممكن وين شلون تقدر تبي تبغى تقدرين تبين تبغين".split())
SKEPTIC_CUES = set(norm(w) for w in "طبعا اكيد أكيد ايه".split())
LAUGH_RE = re.compile(r"(ه){3,}|خخخ|😂|🤣")
YAKOON = set(norm(w) for w in "يكون تكون يكونون يكونوا تكونون".split())

def formula_token_span(host, k=0):
    """Indices (in words(host)) covered by the k-th formula occurrence in the sentence (0 = first), or None."""
    toks = words(host)
    joined, starts = "", []
    for t in toks:
        starts.append(len(joined)); joined += t + " "
    occs = list(find_formula(joined))
    if not occs:
        return toks, None
    s, e, _ = occs[min(k, len(occs) - 1)]
    cover = [i for i, st in enumerate(starts) if st < e and st + len(toks[i]) > s]
    return toks, (cover[0], cover[-1])

def fill_position(host, span=None):
    toks, sp = formula_token_span(host) if span is None else span
    if sp is None:
        return "middle"
    before, after = sp[0] > 0, sp[1] < len(toks) - 1
    if not before and not after:
        return "alone"
    if not before:
        return "first"
    if not after:
        return "last"
    return "middle"

def is_first_person_future(tok, prev_tok):
    return any(_fp_future(t, prev_tok, stripped=i > 0) for i, t in enumerate(forms(tok)))

def _fp_future(t, prev_tok, stripped=False):
    # The guide explicitly identifies بروح as speaker-future; GLF misreads its reduced stem as imperative.
    if t == "بروح":
        return True
    if prev_tok is not None and norm(prev_tok) in FUT_WORDS and t[:1] in ("ا", "ن"):
        return True
    if t.startswith("ب") and len(t) >= 4 and not t.startswith("بال") and t[1] not in "تي" and t not in BA_STOP:
        an = analyses(t)
        if any(p == "verb" and prc1 == "b_fut" and per == "1" for p, _, per, prc1, _ in an):
            return True
    if t[:1] in ("ا", "ن") and len(t) >= (4 if stripped else 3) and not t.startswith("ال"):   # فائز -> ائز is not a verb
        an = analyses(t)   # every reading must be a verb (drops nouns such as اقل، ابناء), one of them 1st-person imperfective
        if an and all(p == "verb" for p, *_ in an) and any(asp == "i" and per == "1" for _, asp, per, _, _ in an):
            return True
    return False

def is_first_person_past(tok):
    return any(_fp_past(t) for t in forms(tok))

def _fp_past(t):
    if len(t) < 3 or t in SECOND_PERSON_WORDS or not (t.endswith("ت") or t.endswith("نا")):
        return False
    return any(p == "verb" and asp == "p" and per == "1" for p, asp, per, _, _ in analyses(t))

def is_verb(tok):
    if norm(tok) in NOT_VERB:
        return False
    an = analyses(norm(tok))
    return bool(an) and sum(p == "verb" for p, *_ in an) / len(an) >= 0.5

def fill_verb_cue(host, span=None):
    toks, sp = formula_token_span(host) if span is None else span
    if sp is None:
        return "none"
    window = list(range(max(0, sp[0] - 3), sp[0])) + list(range(sp[1] + 1, min(len(toks), sp[1] + 4)))
    if any(is_first_person_future(toks[i], toks[i - 1] if i > 0 else None) for i in window):
        return "me_future"
    if any(is_first_person_past(toks[i]) for i in window):
        return "me_past"
    if any(is_verb(toks[i]) for i in window if norm(toks[i]) not in FUT_WORDS):
        return "other"
    return "none"

def fill_future_marker(host):
    toks = words(host)
    for i, tok in enumerate(toks):
        t = norm(tok)
        if t in FUT_WORDS:
            nxt = norm(toks[i + 1]) if i + 1 < len(toks) else ""
            if nxt and not nxt.startswith("ال"):
                return True
        if t.startswith("ب") and len(t) >= 4 and not t.startswith("بال") and t not in BA_STOP:
            if not next(find_formula(tok), None):
                an = analyses(t)
                if any(p == "verb" and prc1 == "b_fut" for p, _, _, prc1, _ in an):
                    return True
    return False

def fill_second_person(host):
    toks = words(host)
    for i, tok in enumerate(toks):
        if any(_second_person_word(t) for t in forms(tok)):
            return True
        t = norm(tok)
        if t in ("يا", "ي") and i + 1 < len(toks) and norm(toks[i + 1]) not in VOCATIVE_GOD:
            return True
        if t.startswith("يا") and len(t) > 3 and t[2:] not in VOCATIVE_GOD and t not in ("يارب", "ياالله", "يالله"):
            return True
    return False

def _second_person_word(t):
        if t in SECOND_PERSON_WORDS:
            return True
        if t.startswith("ال") or next(find_formula(t), None):
            return False
        if len(t) >= 4 and (t.endswith("كم") or t.endswith("كي")) and t not in KUM_KI_EXCEPT and t not in K_EXCEPT:
            return True
        if len(t) >= 3 and t.endswith("ك") and t not in K_EXCEPT:
            an = analyses(t)
            if not an or any(enc0 and "2" in str(enc0) for *_, enc0 in an):
                return True
        return False

# The worked example from the Phase 2 guide must come out exactly as the guide shows.
ex = "ان شاء الله بروح المستشفى بكرة"
assert fill_position(ex) == "first"
assert fill_verb_cue(ex) == "me_future"
assert fill_future_marker(ex) is True
assert fill_second_person(ex) is False
# A few more behaviours we rely on
assert fill_position("طيب ان شاء الله") == "last"
assert fill_position("ان شاء الله") == "alone"
assert fill_verb_cue("رحت للدكتور وقال ان شاء الله خير") in ("me_past", "other")
assert fill_second_person("الله يشفيك ان شاء الله تتحسن") is True
assert fill_second_person("ما شاء الله تبارك الله") is False
assert fill_second_person("ان شاء الله يا رب") is False
assert fill_future_marker("ان شاء الله بكرة الصبح") is False
assert fill_second_person("بارك الله فيكم ان شاء الله") is True
assert fill_verb_cue("ان شاء الله راح اسوي الواجب") == "me_future"
assert fill_verb_cue("ان شاء الله يا رب") == "none"
print("Worked example from the guide reproduced ✔")

## B5. Quote detection (Quran and fixed prayers)
* **Quran:** the host sentence is compared with the Tanzil text on 4-word windows. The formula itself appears in six verses
  (2:70, 12:99, 18:69, 28:27, 37:102, 48:27), so a match only counts when the matched run **contains the formula and at least two more Quranic words**.
  Ordinary uses such as *ان شاء الله من بكرة* are therefore not flagged.
* **Fixed prayers:** apart from the formula, the host sentence is (almost) only a phrase from `FIXED_PRAYERS` — at most 2 other words (spaces ignored, so ماشاء الله also matches). A prayer somewhere inside a long sentence does not flag the row.
Flagged rows stay in D2.1 and are excluded from D2.2.

In [ ]:
CANON = ["ان", "شاء", "الله"]

def canon_tokens(host):
    """Normalized word tokens, with every formula occurrence replaced by the 3 canonical tokens. Returns (tokens, formula_index_sets)."""
    toks = words(host)
    out, fsets, i = [], [], 0
    joined, starts = "", []
    for t in toks:
        starts.append(len(joined)); joined += t + " "
    occs = [(s, e) for s, e, _ in find_formula(joined)]
    cover = []
    for s, e in occs:
        cover.append([k for k, st in enumerate(starts) if st < e and st + len(toks[k]) > s])
    covered = {k: n for n, c in enumerate(cover) for k in c}
    k = 0
    while k < len(toks):
        if k in covered:
            n = covered[k]
            fsets.append(set(range(len(out), len(out) + 3)))
            out.extend(CANON)
            k = cover[n][-1] + 1
        else:
            out.append(norm(toks[k])); k += 1
    return out, fsets

def load_quran_kgrams(path, k=QUOTE_K):
    grams = set()
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        parts = line.split("|")
        if len(parts) != 3:
            continue
        toks = [norm(t) for t in parts[2].split()]
        grams.update(tuple(toks[i:i + k]) for i in range(len(toks) - k + 1))
    return grams

PRAYERS_NOSPACE = [norm(p).replace(" ", "") for p in FIXED_PRAYERS]

def is_fixed_prayer(toks, fsets, max_other=2):
    """True when, apart from the formula, the sentence is (almost) only a fixed prayer: <= max_other other words."""
    formula_idx = set().union(*fsets) if fsets else set()
    covered, found = set(), False
    for i in range(len(toks)):
        joined = ""
        for j in range(i, min(len(toks), i + 10)):
            joined += toks[j]
            if joined in PRAYERS_NOSPACE:
                covered |= set(range(i, j + 1)); found = True
    other = [i for i in range(len(toks)) if i not in covered and i not in formula_idx]
    return found and len(other) <= max_other

def quote_flag_for(host, quran_grams, k=QUOTE_K, min_extra=QUOTE_MIN_EXTRA, occ=0):
    toks, fsets = canon_tokens(host)
    if is_fixed_prayer(toks, fsets):
        return True
    if not fsets:
        return False
    target = fsets[min(occ, len(fsets) - 1)]
    all_formula = set().union(*fsets)
    covered = [False] * len(toks)
    for i in range(len(toks) - k + 1):
        if tuple(toks[i:i + k]) in quran_grams:
            for j in range(i, i + k):
                covered[j] = True
    i = 0
    while i < len(toks):
        if covered[i]:
            j = i
            while j < len(toks) and covered[j]:
                j += 1
            run = set(range(i, j))
            if target <= run and len(run - all_formula) >= min_extra:   # Quranic words other than the formula itself
                return True
            i = j
        else:
            i += 1
    return False

# Test with a tiny stand-in for the Quran (the real file is loaded in Part C)
_mini_quran = INTERIM / "_test_quran.txt"
_mini_quran.write_text("18|69|قال ستجدني إن شاء الله صابرا ولا أعصي لك أمرا\n37|102|ستجدني إن شاء الله من الصابرين\n", encoding="utf-8")
_g = load_quran_kgrams(_mini_quran)
assert quote_flag_for("قال ستجدني ان شاء الله صابرا", _g) is True
assert quote_flag_for("ان شاء الله من بكرة ابدأ الدوام", _g) is False
assert quote_flag_for("ماشاء الله تبارك الله ان شاء الله", _g) is True
assert quote_flag_for("ماشاء الله تبارك الله مطلوب رقمك ع الخاص وجاد بالشراء ان شاء الله", _g) is False   # prayer elsewhere in a long sentence
assert quote_flag_for("ان شاء الله ان شاء الله تعود الروح", _g) is False                                 # repeated formula is not a quote
assert quote_flag_for("ان شاء الله بروح المستشفى بكرة", _g) is False
_mini_quran.unlink()
print("Quote detection tests passed ✔")

## B6. Helpers: dialect markers, stratum cues, IDs, CSV writing

In [ ]:
# Words that point to a NON-Saudi dialect (professor feedback item 8: an example with علشانك is Egyptian).
NON_SAUDI_MARKERS = {
    "Egyptian": "علشان علشانك عشانك_ ازاي دلوقتي دلوقت كده كدا اوي خالص بتاع بتاعك بتاعتي ده دي مش امتى ليه_",
    "Levantine": "هلق هلأ هيك هيدا منيح كتير بدي بدك شو_ لسا",
    "Maghrebi": "بزاف واش ديال دابا برشا",
    "Iraqi": "شكو هواية اكو_ ماكو_ ديگول",
}
_NS = {d: {norm(w) for w in ws.split() if not w.endswith("_")} for d, ws in NON_SAUDI_MARKERS.items()}

def non_saudi_markers(text):
    toks = {norm(strip_conj(t)) for t in words(text or "")}
    hits = [f"{d}:{w}" for d, ws in _NS.items() for w in sorted(ws & toks)]
    return ";".join(hits)

assert non_saudi_markers("ان شاء الله علشانك") == "Egyptian:علشانك"

def has_skeptic_cue(*texts):
    for t in texts:
        if not isinstance(t, str):
            continue
        if LAUGH_RE.search(t) or ({norm(w) for w in words(t)} & SKEPTIC_CUES):
            return True
    return False

def answers_question(prev):
    if not isinstance(prev, str) or not prev.strip():
        return False
    return prev.strip().endswith(("?", "؟")) or bool({norm(w) for w in words(prev)} & QUESTION_WORDS)

def has_yakoon(host):
    return bool({norm(strip_conj(w)) for w in words(host)} & YAKOON)

def write_csv(df, path):
    """UTF-8 with BOM so Arabic opens correctly in Excel."""
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"  wrote {path.relative_to(PROJECT_DIR)}  ({len(df):,} rows)")

def read_csv(path):
    return pd.read_csv(path, encoding="utf-8-sig", dtype=str, keep_default_na=False)

def sha1(s):
    return hashlib.sha1(s.encode("utf-8")).hexdigest()

def wilson(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0, c - h), min(1, c + h))

print("Library ready ✔")

def occurrence_key(source, original_sha1, start, end, record_ref=""):
    return sha1(f"{source}|{record_ref}|{original_sha1}|{int(start)}|{int(end)}")

def apply_verified_corrections(frame, reviewed, fields):
    out = frame.copy()
    logs = []
    if reviewed.empty or "occurrence_key" not in reviewed:
        return out, pd.DataFrame(logs, columns=["occurrence_key", "id", "field", "before", "after"])
    valid = reviewed[reviewed["reviewed"].str.strip().str.lower().isin(["y", "yes", "1", "x"])]
    if valid["occurrence_key"].duplicated().any():
        raise ValueError("Duplicate occurrence keys in the review sheet")
    lookup = valid.set_index("occurrence_key")
    # Iterate only the reviewed rows, rather than the whole full-size master.
    for idx, row in out[out["occurrence_key"].isin(lookup.index)].iterrows():
        key = row["occurrence_key"]
        if key not in lookup.index:
            continue
        r = lookup.loc[key]
        if any(str(row[col]) != str(r[col]) for col in ["host_sentence", "prev_context"]):
            continue  # changed context requires fresh review
        for field in fields:
            value = str(r.get("correct_" + field, "")).strip()
            if value:
                allowed = {"position": {"first", "middle", "last", "alone"},
                           "verb_cue": {"me_future", "me_past", "other", "none"},
                           "future_marker": {"true", "false"}, "second_person": {"true", "false"},
                           "quote_flag": {"true", "false"}}
                if field in allowed and value not in allowed[field]:
                    raise ValueError(f"Invalid manual correction for {field}: {value}")
                if field == "variant" and value not in row.get("original_text", ""):
                    raise ValueError("Corrected variant is absent from original_text")
                if str(row[field]) != value:
                    logs.append(dict(occurrence_key=key, id=row["id"], field=field, before=row[field], after=value))
                    out.at[idx, field] = value
    return out, pd.DataFrame(logs, columns=["occurrence_key", "id", "field", "before", "after"])

def match_sheet(sheet, current):
    """Resolve prior review by occurrence identity, never by sequential CSV ID."""
    if sheet.empty or "occurrence_key" not in sheet:
        return sheet.head(0).copy()
    if sheet["occurrence_key"].duplicated().any():
        raise ValueError("Duplicate occurrence keys in a filled sheet")
    cur = current.set_index("occurrence_key")
    rows = []
    for _, r in sheet.iterrows():
        key = r["occurrence_key"]
        if key not in cur.index:
            continue
        c = cur.loc[key]
        if r["host_sentence"] != c["host_sentence"] or r["prev_context"] != c["prev_context"]:
            continue
        d = r.to_dict()
        d.update(id=c["id"], source=c["source"])
        rows.append(d)
    return pd.DataFrame(rows, columns=sheet.columns)

def plan_quotas(sources, available, rates, source_totals, target=50, floor=1):
    """Integer source-by-stratum allocation. Returns None if the constraints cannot be met."""
    from scipy.optimize import milp, Bounds, LinearConstraint
    sources, available, rates = list(sources), np.asarray(available), np.asarray(rates, dtype=float)
    size = len(sources)
    if not size:
        return None
    rows, lo, hi = [], [], []
    for source, count in source_totals.items():
        rows.append([int(s == source) for s in sources]); lo.append(count); hi.append(count)
    for label in range(rates.shape[1]):
        rows.append(rates[:, label]); lo.append(target); hi.append(np.inf)
    lower = np.minimum(available, floor)
    # Prefer broad coverage and avoid unneeded domination by high-WISH cells.
    cost = -np.sum(rates[:, [2, 4]], axis=1) + np.arange(size) * 1e-8
    res = milp(c=cost, integrality=np.ones(size), bounds=Bounds(lower, available),
               constraints=LinearConstraint(np.asarray(rows), np.asarray(lo), np.asarray(hi)),
               options={"time_limit": 20, "mip_rel_gap": 0.0})
    if res.x is None:
        return None
    allocation = np.rint(res.x).astype(int)
    if any(sum(allocation[i] for i, s in enumerate(sources) if s == source) != count for source, count in source_totals.items()):
        return None
    if (rates.T @ allocation < target - 1e-7).any():
        return None
    return allocation

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

---
# Part C — Phase 1, item 5: data, access log, extraction prototype, real examples

## C1. Download the three datasets and the Quran text
* **SFC-mini** (Hugging Face, 63 text files, ~3 GB, one post per line).
* **Gulf Arabic Tweets 2018–2020** (Hugging Face, one CSV, ~108k tweets).
* **SADA** text fields. Default: the Kaggle CSVs, because they keep the recording name and segment start time, which we need for the *previous segment*. The audio is **not** downloaded.
  Kaggle needs an API token: in Colab open the 🔑 *Secrets* panel and add `KAGGLE_USERNAME` and `KAGGLE_KEY` (from kaggle.com → Settings → API → Create New Token).
  If you cannot use Kaggle, set `SADA_SOURCE = "hf"`; the text columns are then read from the Hugging Face copy without the audio, but the previous segment is only approximate (row order).
* **Tanzil Quran text** (simple-clean, `sura|aya|text`), for quote detection.

Files that are already present are not downloaded again.

In [ ]:
from huggingface_hub import HfApi, hf_hub_download
# Optional authentication stays in Colab Secrets / environment, never source.
if IN_COLAB and not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        secret_token = userdata.get("HF_TOKEN")
        if secret_token:
            os.environ["HF_TOKEN"] = secret_token
        del secret_token
    except Exception:
        pass  # these source repositories are public
api = HfApi()

def hf_meta(repo, suffix, local_dir):
    """(revision, files) from the Hub; if the Hub is unreachable, fall back to files already on disk."""
    try:
        pinned = {SFC_REPO: 'cae145d641901698f068f806ab27d95fba042ab9', GULF_REPO: 'b549bc3dc6e25ca2980695d082de2818ec9733f2'}.get(repo)
        # Resolve the requested revision; log the full SHA returned by the Hub.
        rev = api.dataset_info(repo, revision=pinned).sha
        files = [f for f in api.list_repo_files(repo, repo_type="dataset", revision=rev) if f.endswith(suffix)]
        return rev, files
    except Exception as e:
        local = sorted(str(p.relative_to(local_dir)) for p in local_dir.rglob(f"*{suffix}"))
        print(f"  Hub not reachable for {repo} ({type(e).__name__}); using {len(local)} local files")
        return "unknown (offline)", local
ACCESS = []   # rows for the data access log

def log_access(name, link, licence, files, rows, revision="", note=""):
    ACCESS.append(dict(source=name, link=link, licence=licence, files=files, rows=rows, revision=revision,
                       accessed_utc=dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%d %H:%M"), status="downloaded", note=note))

# --- SFC-mini --------------------------------------------------------------
sfc_dir = RAW / "sfc_mini"; sfc_dir.mkdir(parents=True, exist_ok=True)
sfc_rev, sfc_files = hf_meta(SFC_REPO, ".txt", sfc_dir)
sfc_files = sorted(sfc_files, key=lambda f: int(re.sub(r"\D", "", Path(f).stem) or 0))
if SFC_MAX_FILES:
    sfc_files = sfc_files[:SFC_MAX_FILES]
print(f"SFC-mini: {len(sfc_files)} files to use")
for f in sfc_files:
    if not (sfc_dir / f).exists():
        hf_hub_download(SFC_REPO, f, repo_type="dataset", revision=sfc_rev, local_dir=sfc_dir)
print("SFC-mini ready")

# --- Gulf tweets -------------------------------------------------------------
gulf_dir = RAW / "gulf_tweets"; gulf_dir.mkdir(parents=True, exist_ok=True)
gulf_rev, gulf_csvs = hf_meta(GULF_REPO, ".csv", gulf_dir)
gulf_csv_name = gulf_csvs[0]
if not (gulf_dir / gulf_csv_name).exists():
    hf_hub_download(GULF_REPO, gulf_csv_name, repo_type="dataset", revision=gulf_rev, local_dir=gulf_dir)
print("Gulf tweets ready:", gulf_csv_name)

# --- Tanzil -------------------------------------------------------------------
tanzil_path = RAW / "tanzil" / "quran-simple-clean.txt"; tanzil_path.parent.mkdir(parents=True, exist_ok=True)
if not tanzil_path.exists():
    import urllib.request
    try:
        urllib.request.urlretrieve(TANZIL_URL, tanzil_path)
    except Exception as e:
        raise SystemExit(f"Could not download Tanzil ({e}). Download 'Simple Clean' as 'Text (with aya numbers)' from "
                         f"https://tanzil.net/download/ and upload it to {tanzil_path}")
n_ayat = sum(1 for l in tanzil_path.read_text(encoding="utf-8").splitlines() if l.count("|") == 2)
if n_ayat != 6236:
    raise ValueError(f"Tanzil validation failed: expected 6236 verses, got {n_ayat}. Replace the file before continuing.")
print("Tanzil ready:", n_ayat, "verses")

In [ ]:
# --- SADA ---------------------------------------------------------------------
sada_dir = RAW / "sada"; sada_dir.mkdir(parents=True, exist_ok=True)
SADA_TEXT_COLS = ["FileName", "ShowName", "SegmentID", "SegmentStart", "SegmentEnd", "SpeakerDialect", "Speaker",
                  "GroundTruthText", "ProcessedText", "Category"]

def get_sada_kaggle():
    import zipfile, shutil
    if IN_COLAB:
        try:
            from google.colab import userdata
            os.environ.setdefault("KAGGLE_USERNAME", userdata.get("KAGGLE_USERNAME"))
            os.environ.setdefault("KAGGLE_KEY", userdata.get("KAGGLE_KEY"))
        except Exception as e:
            print("Kaggle secrets not found in Colab:", e)
    paths = []
    for split in ["train.csv", "valid.csv", "test.csv"]:
        target = sada_dir / split
        if not target.exists() or zipfile.is_zipfile(target):
            import kagglehub
            try:
                p = target if target.exists() else kagglehub.dataset_download(f"{SADA_KAGGLE}/versions/23", path=split)
            except Exception as e:
                raise SystemExit(f"Kaggle download of {split} failed ({e}).\nEither add KAGGLE_USERNAME / KAGGLE_KEY to Colab Secrets, "
                                 f"or download train.csv, valid.csv and test.csv from https://www.kaggle.com/datasets/{SADA_KAGGLE} "
                                 f"and upload them to {sada_dir}.")
            # Kaggle may return a ZIP even for a single-file request.
            if zipfile.is_zipfile(p):
                temporary = target.with_suffix(".csv.tmp")
                with zipfile.ZipFile(p) as archive:
                    member = next(name for name in archive.namelist() if Path(name).name == split)
                    with archive.open(member) as incoming, temporary.open("wb") as outgoing:
                        shutil.copyfileobj(incoming, outgoing)
                temporary.replace(target)
            else:
                shutil.copyfile(p, target)
            required = {"FileName", "SegmentStart", "SegmentEnd", "GroundTruthText"}
            assert required.issubset(pd.read_csv(target, nrows=0).columns), f"Invalid SADA CSV: {target}"
        paths.append(target)
    return paths

def get_sada_hf():
    target = sada_dir / "sada_hf_text.parquet"
    if not target.exists():
        import pyarrow.parquet as pq
        from huggingface_hub import HfFileSystem
        fs = HfFileSystem()
        _, files = hf_meta(SADA_HF_REPO, ".parquet", sada_dir)
        parts = []
        for f in files:   # read only the text columns: the audio column is never downloaded
            with fs.open(f"datasets/{SADA_HF_REPO}/{f}") as fh:
                t = pq.read_table(fh, columns=["text", "cleaned_text", "speaker_dialect"]).to_pandas()
            t["hf_file"], t["hf_row"] = f, range(len(t))
            parts.append(t)
        pd.concat(parts).to_parquet(target)
    return [target]

if SADA_SOURCE == "kaggle":
    sada_paths = get_sada_kaggle()
else:
    raise ValueError("Use SADA_SOURCE='kaggle' or upload the three source CSVs. The text-only HF fallback cannot establish previous-segment order.")
print("SADA ready:", [p.name for p in sada_paths])

## C2. Load the records and look at their structure
One **record** = one forum post (SFC-mini line), one tweet, or one SADA segment. Each record gets a reference (`file:line` or `file:row`) so every example can be traced back to the source.

In [ ]:
def iter_sfc():
    for f in sfc_files:
        path = sfc_dir / f
        with open(path, encoding="utf-8", errors="strict") as fh:
            for ln, line in enumerate(fh, 1):
                line = line.rstrip("\n")
                if line.strip():
                    yield f"{f}:{ln}", line

gulf_df = pd.read_csv(gulf_dir / gulf_csv_name, dtype=str, keep_default_na=False)
GULF_TEXT_COL = "text" if "text" in gulf_df.columns else gulf_df.columns[0]
gulf_df["record_ref"] = [f"{gulf_csv_name}:{i}" for i in range(len(gulf_df))]

if SADA_SOURCE == "kaggle":
    frames = []
    for p in sada_paths:
        d = pd.read_csv(p, dtype=str, keep_default_na=False)
        d["split_file"] = p.name
        d["row_in_file"] = range(len(d))
        frames.append(d)
    sada_df = pd.concat(frames, ignore_index=True)
    SADA_TEXT_COL = "GroundTruthText" if "GroundTruthText" in sada_df.columns else "ProcessedText"
    sada_df["_start"] = pd.to_numeric(sada_df["SegmentStart"], errors="coerce")
    sada_df = sada_df.sort_values(["FileName", "_start"], kind="stable").reset_index(drop=True)
    sada_df["recording"] = sada_df["FileName"]
    sada_df["dialect"] = sada_df.get("SpeakerDialect", "")
    sada_df["record_ref"] = sada_df["split_file"] + ":" + sada_df["row_in_file"].astype(str)
else:
    sada_df = pd.read_parquet(sada_paths[0])
    SADA_TEXT_COL = "text"
    sada_df["recording"] = sada_df["hf_file"]
    sada_df["dialect"] = sada_df["speaker_dialect"]
    sada_df["record_ref"] = sada_df["hf_file"] + ":" + sada_df["hf_row"].astype(str)
sada_df[SADA_TEXT_COL] = sada_df[SADA_TEXT_COL].fillna("").astype(str)

print("Gulf columns:", list(gulf_df.columns)[:6], "| rows:", f"{len(gulf_df):,}")
print("SADA columns:", list(sada_df.columns)[:16], "| rows:", f"{len(sada_df):,}", "| text column:", SADA_TEXT_COL)
print("\nSADA speaker dialects:\n", sada_df["dialect"].value_counts().head(12).to_string())
print("\n--- 3 SFC-mini records ---")
for i, (ref, t) in enumerate(iter_sfc()):
    print(ref, "|", t[:150]);
    if i == 2: break
print("--- 3 Gulf tweets ---\n" + "\n".join(gulf_df[GULF_TEXT_COL].head(3).str[:150]))
print("--- 3 SADA segments ---\n" + "\n".join(sada_df[SADA_TEXT_COL].head(3).str[:150]))

## C3. Extraction (used by Phase 1 and by Phase 2, Steps 1–2)
Every record is searched; **each occurrence becomes one row**. Two occurrences in one post give two rows.
This is the slow cell for SFC-mini (about 8.9 million posts; expect roughly 10–20 minutes on Colab). A cheap pre-filter (ن followed by ش + alef, diacritics allowed) skips most posts before the full search.

In [ ]:
from tqdm.auto import tqdm
# Match the same removable marks as normalize_with_map, including marks after whitespace and alef-wasla.
PREFILTER = re.compile(r"ن[\sؐ-ًؚ-ٰٟۖ-ۭـ]*ش[ؐ-ًؚ-ٰٟۖ-ۭـ]*[اآأإٱ]")
assert all(PREFILTER.search(t) for t, v in _tests.items() if v), "pre-filter would miss a spelling"
N_RECORDS = Counter()

def occurrences(source, record_ref, text, prev_record_text="", recording="", dialect=""):
    N_RECORDS[source] += 1
    rows = []
    if not PREFILTER.search(text):
        return rows
    spans = sentence_spans(text)
    for k, (start, end, variant) in enumerate(find_formula(text)):
        si = next((i for i, (a, b) in enumerate(spans) if a <= start < b), None)
        host_raw = text[spans[si][0]:spans[si][1]] if si is not None else text
        if source == "sada":
            prev_raw = prev_record_text
        elif si is not None and si > 0:
            prev_raw = text[spans[si - 1][0]:spans[si - 1][1]]      # previous segment of the same recording
        else:
            prev_raw = ""                    # never take context from another post or tweet
        h0 = spans[si][0] if si is not None else 0
        occ_in_host = sum(1 for s2, _, _ in find_formula(host_raw) if s2 < start - h0)   # 2nd formula in the same sentence -> 1
        rows.append(dict(source=source, record_ref=record_ref, occurrence=k, occ_in_host=occ_in_host, char_start=start, char_end=end,
                         original_text=text, original_sha1=sha1(text), host_raw=host_raw, prev_raw=prev_raw,
                         variant=variant, recording=recording, dialect=dialect, segment_prev_raw=prev_record_text))
    return rows

raw_rows = []
for ref, text in tqdm(iter_sfc(), desc="SFC-mini", unit=" posts", mininterval=5):
    raw_rows += occurrences("sfc", ref, text)
for ref, text in tqdm(zip(gulf_df["record_ref"], gulf_df[GULF_TEXT_COL]), total=len(gulf_df), desc="Gulf"):
    raw_rows += occurrences("gulf", ref, text)
prev_text, prev_rec = "", None
for ref, text, rec, dia in tqdm(zip(sada_df["record_ref"], sada_df[SADA_TEXT_COL], sada_df["recording"], sada_df["dialect"]),
                                total=len(sada_df), desc="SADA"):
    raw_rows += occurrences("sada", ref, text, prev_text if rec == prev_rec else "", rec, dia)
    prev_text, prev_rec = text, rec

raw = pd.DataFrame(raw_rows)
del raw_rows  # release per-row dictionaries after constructing the dataframe
write_csv(raw, INTERIM / "raw_hits.csv")
print("\nRaw occurrences per source:\n", raw["source"].value_counts().to_string())

In [ ]:
# Records read per source, for the access log
n_sfc_records = N_RECORDS["sfc"]
log_access("SFC-mini", f"https://huggingface.co/datasets/{SFC_REPO}", "CC BY-NC 4.0",
           f"{len(sfc_files)} txt files", n_sfc_records, sfc_rev, "one post per non-empty line")
log_access("Gulf Arabic Tweets 2018-2020", f"https://huggingface.co/datasets/{GULF_REPO}", "CC BY 4.0",
           gulf_csv_name, len(gulf_df), gulf_rev, "text is pre-cleaned by the dataset authors (no URLs, user names or diacritics)")
if SADA_SOURCE == "kaggle":
    log_access("SADA (text fields only)", f"https://www.kaggle.com/datasets/{SADA_KAGGLE}", "CC BY-NC-SA 4.0",
               ", ".join(p.name for p in sada_paths), len(sada_df), "", f"text column {SADA_TEXT_COL}; audio not downloaded")
else:
    log_access("SADA (text fields only)", f"https://huggingface.co/datasets/{SADA_HF_REPO}", "CC BY-NC-SA 4.0",
               "parquet text columns", len(sada_df), hf_meta(SADA_HF_REPO, ".parquet", sada_dir)[0], "audio not downloaded; segment order approximate")
log_access("Tanzil Quran text (simple-clean)", "https://tanzil.net/download/", "CC BY 3.0 (verbatim copying only)",
           tanzil_path.name, n_ayat, "", "used only for quote detection")
ACCESS.append(dict(source="SaudiIrony", link="", licence="unknown", files="", rows="", revision="", accessed_utc="",
                   status="not used (not one of the three named sources; ask before using)", note=""))
access_df = pd.DataFrame(ACCESS)
write_csv(access_df, REP1 / "data_access_log.csv")
lines = ["# D1.3 Data access log", "", f"Generated by the notebook on {dt.date.today()}.", "",
         "| Source | Link | Licence | Files | Records | Revision | Accessed (UTC) | Status | Note |", "|---|---|---|---|---|---|---|---|---|"]
for r in ACCESS:
    n = f"{r['rows']:,}" if isinstance(r["rows"], (int, np.integer)) else str(r["rows"])
    lines.append(f"| {r['source']} | {r['link']} | {r['licence']} | {r['files']} | {n} | {str(r['revision'])[:12]} | {r['accessed_utc']} | {r['status']} | {r['note']} |")
lines += ["", "**What can be released:** SFC-mini (NC) and SADA (NC-SA) restrict commercial use, and SADA adds share-alike. "
          "A release of the annotated data should therefore be non-commercial and carry the SADA share-alike terms, "
          "or contain only IDs and labels. Confirm with the instructor."]
(REP1 / "data_access_log.md").write_text("\n".join(lines), encoding="utf-8")
print(access_df[["source", "licence", "rows", "status"]].to_string(index=False))

## C4. Extraction prototype results (for D1.2 and D1.3)
How often each spelling occurs in each source, before any cleaning.

In [ ]:
proto = raw.groupby(["source", "variant"]).size().rename("count").reset_index().sort_values(["source", "count"], ascending=[True, False])
write_csv(proto, REP1 / "prototype_variant_counts.csv")
print(proto.groupby("source").head(8).to_string(index=False))

## C5. Candidate examples for each label (D1.2)
Every example in the label inventory must be **real, from the corpus, with source and ID**.
This cell finds likely candidates for each label with simple cues and writes them to `manual/phase1_label_candidates.csv`.
The cues only help you find examples; **you** decide the label.

* Rows with a `non_saudi_markers` value (for example *Egyptian:علشانك*) should not be chosen (feedback item 8). For SADA, prefer Saudi speaker dialects.
* **What you do:** open the file, put `y` in the `chosen` column for at least 3 rows per label (you can change `label` if a row fits another label better), save it as `manual/phase1_label_examples_filled.csv`, upload it, and run C6.

In [ ]:
def light_clean(df):
    d = df.copy()
    d["host_sentence"] = d["host_raw"].map(clean_working)
    d["prev_context"] = d["prev_raw"].map(clean_working)
    d["n_words"] = d["host_sentence"].map(n_words)
    d = d[(d["n_words"] >= MIN_WORDS) & (d["n_words"] <= MAX_WORDS)]
    return d.drop_duplicates(subset=["host_sentence"]).copy()

cand = light_clean(raw)
# Bound candidate-search work without changing the full collection.
if len(cand) > 30000:
    cand = cand.sample(30000, random_state=SEED)
cand["verb_cue"] = cand["host_sentence"].map(fill_verb_cue)
cand["position"] = cand["host_sentence"].map(fill_position)
cand["second_person"] = cand["host_sentence"].map(fill_second_person)
cand["non_saudi_markers"] = (cand["host_sentence"] + " " + cand["prev_context"]).map(non_saudi_markers)
nh = lambda s: {norm(w) for w in words(s)}
CUES = {
    "COMMIT":  lambda r: r.verb_cue == "me_future" and not r.second_person,
    "WISH":    lambda r: bool(nh(r.host_sentence) & {norm(w) for w in "يارب عسى نتمنى اتمنى يشفيك يوفقك يسعدك يرحمه يحفظك".split()}),
    "DEFER":   lambda r: answers_question(r.prev_context) and r.verb_cue != "me_future",
    "HEDGE":   lambda r: r.verb_cue == "me_past" or has_yakoon(r.host_sentence),
    "SKEPTIC": lambda r: has_skeptic_cue(r.host_sentence, r.prev_context),
    "CLOSE":   lambda r: r.position == "last" and bool(nh(r.host_sentence) & {norm(w) for w in "طيب تمام خلاص ابشر اوكي حاضر ok".split()}),
    "OTHER":   lambda r: bool(nh(r.host_sentence) & {norm(w) for w in "بتشوف تشوف انتظر حسابك احذر انتبه ترى".split()}),
}
picks = []
for label, rule in CUES.items():
    hit = cand[cand.apply(rule, axis=1)]
    hit = hit.assign(_ns=hit["non_saudi_markers"] != "").sort_values("_ns", kind="stable")   # Saudi-looking rows first
    per_source = [g.sample(min(len(g), 15), random_state=SEED) for _, g in hit[~hit["_ns"]].groupby("source")]
    pick = pd.concat(per_source) if per_source else hit.head(0)
    pick = pick.assign(label=label, cue=label + " cue")
    picks.append(pick)
    print(f"{label:8s} {len(hit):6,} cue matches -> {len(pick)} candidates")
cands = pd.concat(picks, ignore_index=True)
cands["chosen"] = ""
cands_out = cands[["label", "chosen", "source", "record_ref", "dialect", "non_saudi_markers", "variant", "host_sentence", "prev_context", "original_text"]]
if not (MANUAL / "phase1_label_candidates.csv").exists():
    write_csv(cands_out, MANUAL / "phase1_label_candidates.csv")

## C6. Label inventory draft (D1.2): definitions, boundary rules and chosen examples
Definitions follow the project brief. The three boundary rules answer feedback item 6, and OTHER answers item 7.
The final inventory is D1_2_Label_Inventory_v0_4.

In [ ]:
DEFINITIONS = {
    "COMMIT":  "The speaker states their own future action (a promise or intention) and is expected to carry it out.",
    "WISH":    "Hope or blessing about an outcome the speaker does not control (health, success, weather, another person's action).",
    "DEFER":   "A non-committal or evasive reply to a request or question; the speaker avoids committing to the action.",
    "HEDGE":   "Epistemic softening about a present or past state the speaker is unsure of (\"it should be fine, God willing\").",
    "SKEPTIC": "Sarcastic or doubting use: the speaker signals that someone else's claim, promise or expected event will probably not happen.",
    "CLOSE":   "Formulaic acknowledgement or topic closing; no new hoped-for outcome is added.",
    "OTHER":   "A recognisable function outside the six labels, such as a warning or threat (\"you'll see, God willing\"). Write the function in a note.",
    "UNCLEAR": "Only when the available context is not enough to decide. Not for functions outside the label set (use OTHER).",
    "QUOTE":   "Quranic quotation or fixed prayer. Flagged automatically (quote_flag) and excluded from annotation.",
}
BOUNDARY_RULES = [
    ("DEFER vs SKEPTIC", "Ask whose action or claim is in doubt. If the speaker answers a request about THEIR OWN action without committing, it is DEFER. "
                         "If the speaker reacts to SOMEONE ELSE's promise, claim or expected event and signals disbelief (irony cues such as طبعا، ايه، ههه), it is SKEPTIC."),
    ("WISH vs CLOSE",    "Try to paraphrase as \"I hope that X\". If X is a concrete outcome named or clearly recoverable in the sentence or the previous sentence, it is WISH. "
                         "If X would be empty (the formula only acknowledges, agrees or ends the exchange, e.g. تمام ان شاء الله), it is CLOSE."),
    ("COMMIT vs DEFER",  "Both concern the speaker's own future action. Ask: would the speaker be seen as breaking a promise if they did not do it? "
                         "Yes (a specific action is stated, often with a time, and no avoidance signal) -> COMMIT. No (the action is not stated or is left open: نشوف، بعدين، إذا قدرت, or a bare formula as the whole answer) -> DEFER."),
]

ex_filled = MANUAL / "phase1_label_examples_filled.csv"
inv = ["# D1.2 Label inventory v0.2", "", "Examples are real corpus rows chosen in the notebook; each has its source and record reference.", "",
       "## Definitions", ""]
for lab, d in DEFINITIONS.items():
    inv.append(f"- **{lab}** — {d}")
inv += ["", "## Boundary rules for close pairs", ""] + [f"- **{a}.** {b}" for a, b in BOUNDARY_RULES]
inv += ["", "## Examples", ""]
if ex_filled.exists():
    ch = read_csv(ex_filled)
    ch = ch[ch["chosen"].str.strip().str.lower().isin(["y", "yes", "1", "x"])]
    bad = ch[ch["non_saudi_markers"].str.strip() != ""]
    if len(bad):
        print(f"⚠ {len(bad)} chosen examples contain non-Saudi markers — please replace them:")
        print(bad[["label", "source", "record_ref", "non_saudi_markers"]].to_string(index=False))
    for lab in SIX_LABELS + ["OTHER"]:
        rows = ch[ch["label"] == lab]
        need = "need 3" if lab in SIX_LABELS else "3 recommended for the new label"
        flag = "" if len(rows) >= 3 else f"  ⚠ only {len(rows)} chosen, {need}"
        inv.append(f"### {lab}{flag}")
        inv.append("")
        inv.append("| # | Source | Record | Host sentence | Previous sentence |")
        inv.append("|---|---|---|---|---|")
        for i, r in enumerate(rows.itertuples(), 1):
            inv.append(f"| {i} | {r.source} | {r.record_ref} | {r.host_sentence} | {r.prev_context} |")
        inv.append("")
        print(f"{lab:8s} {len(rows)} examples chosen{flag}")
else:
    inv.append("_No examples chosen yet: fill manual/phase1_label_candidates.csv and save it as manual/phase1_label_examples_filled.csv._")
    print("➡ Waiting for you: choose examples in manual/phase1_label_candidates.csv, save as phase1_label_examples_filled.csv, upload, rerun this cell.")
(REP1 / "label_inventory_v0.2.md").write_text("\n".join(inv), encoding="utf-8")
print("wrote reports/phase1/label_inventory_v0.2.md")

---
# Part D — Phase 2: corpus collection and cleaning
Steps 1 and 2 (search all spellings, one row per occurrence) were done in C3 (`raw_hits.csv`).

## Step 3. Clean and filter
Order: remove links and user names from the working copies → remove exact and near-duplicates (before any sampling) → remove hashtag-only posts → keep host sentences of 4–80 words.
Counts are kept after every stage for the report.

**Duplicate key.** Forum posts and tweets: the normalized full record (so a bare *ان شاء الله* reply in different posts is not merged with all other bare replies by accident; identical retweets are merged).
SADA: host sentence + previous segment. All occurrences inside a kept record are kept.

In [ ]:
from datasketch import MinHash, MinHashLSH

def dedupe_key_text(s):
    s = norm(clean_working(s))
    s = re.sub(rf"[^{AR_LETTER}A-Za-z0-9 ]", " ", s)
    s = re.sub(r"(.)\1{2,}", r"\1", s)
    return re.sub(r"\s+", " ", s).strip()

def exact_duplicate_units(rows):
    """Keep every occurrence in the first retained source record for each normalized key."""
    first = rows.groupby(["source", "dkey"], sort=False)["record_ref"].transform("first")
    return set(rows.loc[rows["record_ref"] != first, "unit"])

def char_shingles(key):
    return {key[i:i + 4] for i in range(max(0, len(key) - 3))}

def jaccard(a, b):
    return len(a & b) / len(a | b) if a or b else 1.0

df = raw.copy()
df["host_sentence"] = df["host_raw"].map(clean_working)
df["prev_context"] = df["prev_raw"].map(clean_working)
stage = {"found": df.groupby("source").size()}
# Record-level SADA key; repeated formula occurrences are kept together.
df["dkey"] = [dedupe_key_text(o + (" || " + p if s == "sada" else ""))
               for o, p, s in zip(df["original_text"], df["segment_prev_raw"], df["source"])]
df["unit"] = df["source"] + "|" + df["record_ref"]
units = df.drop_duplicates("unit")[["unit", "source", "record_ref", "dkey"]].reset_index(drop=True)
exact_dups = exact_duplicate_units(units)
near_dups = set()
for source, keep_units in units[~units["unit"].isin(exact_dups)].groupby("source", sort=True):
    lsh = MinHashLSH(threshold=NEAR_DUP_THRESHOLD, num_perm=64)
    kept_keys = {}
    for u, key in tqdm(zip(keep_units["unit"], keep_units["dkey"]), total=len(keep_units), desc=f"near-duplicates {source}", mininterval=5):
        if len(key) < 20:
            continue
        shingles = char_shingles(key)
        m = MinHash(num_perm=64, seed=SEED)
        m.update_batch([s.encode("utf-8") for s in sorted(shingles)])
        # LSH proposes candidates; exact Jaccard decides removal.
        candidates = lsh.query(m)
        if any(jaccard(shingles, char_shingles(kept_keys[v])) >= NEAR_DUP_THRESHOLD for v in candidates):
            near_dups.add(u)
        else:
            lsh.insert(u, m)
            kept_keys[u] = key
df = df[~df["unit"].isin(exact_dups | near_dups)]
del units, keep_units, kept_keys, lsh
df = df.drop(columns=["dkey", "unit"])
stage["after_dedupe"] = df.groupby("source").size()
print(f"Removed {len(exact_dups):,} exact and {len(near_dups):,} verified near-duplicate records")
df = df[~df["original_text"].map(is_hashtag_only)]
stage["after_hashtag_filter"] = df.groupby("source").size()
df["n_words"] = df["host_sentence"].map(n_words)
df = df[df["n_words"].between(MIN_WORDS, MAX_WORDS)]
df = df[df["host_sentence"].map(lambda s: next(find_formula(s), None) is not None)]
stage["after_all_filters"] = df.groupby("source").size()
full_variants = df.groupby(["source", "variant"]).size().rename("count").reset_index()
write_csv(full_variants, REP2 / "table_full_cleaned_variants.csv")
for src, cap in (D21_MAX_ROWS or {}).items():
    if cap and (df["source"] == src).sum() > cap:
        keep = df[df["source"] == src].sample(cap, random_state=SEED).index
        df = df[(df["source"] != src) | df.index.isin(keep)]
        print(f"TRIAL CAP: {src} retained {cap:,} occurrences; full counts saved separately")
stage["in_D2.1"] = df.groupby("source").size()
stage_df = pd.DataFrame(stage).fillna(0).astype(int)
stage_df.loc["total"] = stage_df.sum()
write_csv(stage_df.reset_index(names="source"), REP2 / "table_counts_by_stage.csv")
print(stage_df.to_string())

**Note on "alone".** The guide defines `position = alone` (nothing else in the sentence) but also requires host sentences of at least 4 words.
The formula is 1–3 words, so a truly alone formula cannot pass the length filter, and `alone` will be (almost) empty in D2.1.
Short replies are still sampled through the `short_or_alone` stratum (formula + at most 2 words). **Ask the instructor** whether alone formulas should be exempt from the 4-word minimum.

In [ ]:
print("Rows dropped only because the host sentence was the bare formula:",
      int((raw.assign(h=raw["host_raw"].map(clean_working)).pipe(lambda d: d[d["h"].map(lambda s: fill_position(s) == "alone")])).shape[0]))

## Step 4. Fill the 11 columns, and quote detection

In [ ]:
quran_grams = load_quran_kgrams(tanzil_path)
print(f"Quran {QUOTE_K}-word windows: {len(quran_grams):,}")

def features(host, k):
    """k = which occurrence of the formula in this sentence the row is about (0 = first)."""
    span = formula_token_span(host, k)
    return pd.Series({"position": fill_position(host, span), "verb_cue": fill_verb_cue(host, span),
                      "future_marker": fill_future_marker(host), "second_person": fill_second_person(host),
                      "quote_flag": quote_flag_for(host, quran_grams, occ=k)})

tqdm.pandas(desc="features", mininterval=5)
feat = df.progress_apply(lambda r: features(r["host_sentence"], int(r["occ_in_host"])), axis=1)
df = pd.concat([df.drop(columns=[c for c in feat.columns if c in df.columns]), feat], axis=1)

# ids: source + running number, in source order (stable across reruns)
df = df.drop(columns=["raw_idx"], errors="ignore").rename_axis("raw_idx").reset_index().sort_values(["source", "raw_idx"], kind="stable").reset_index(drop=True)
df["id"] = df["source"] + "_" + df.groupby("source").cumcount().add(1).astype(str).str.zfill(6)
for c in ["future_marker", "second_person", "quote_flag"]:
    df[c] = df[c].map(lambda b: "true" if b else "false")

df["occurrence_key"] = [occurrence_key(s, h, a, b, ref) for s, h, a, b, ref in
                        zip(df["source"], df["original_sha1"], df["char_start"], df["char_end"], df["record_ref"])]
assert df["occurrence_key"].is_unique
AUTO_BEFORE_REVIEW = df.copy()
filled_check = MANUAL / "D2_check50_filled.csv"
override_path = MANUAL / "D2_metadata_overrides.csv"
override_parts = []
for path in [override_path, filled_check]:
    if path.exists():
        override_parts.append(read_csv(path))
if override_parts:
    overrides = pd.concat(override_parts, ignore_index=True).drop_duplicates("occurrence_key", keep="last")
    write_csv(overrides, override_path)
    df, correction_log = apply_verified_corrections(df, overrides, CHECK_FIELDS)
    write_csv(correction_log, REP2 / "table_applied_corrections.csv")
D21 = df[D21_COLUMNS].copy()
write_csv(D21, PROCESSED / "D2.1_master_collection.csv")
prov = df[["id", "source", "record_ref", "occurrence", "char_start", "char_end", "original_sha1", "recording", "dialect", "n_words", "occurrence_key"]]
write_csv(prov, PROCESSED / "D2.1_provenance.csv")
print("\nquote_flag:", D21["quote_flag"].value_counts().to_dict())
print(D21.groupby("source").size().to_string())

In [ ]:
# original_text must be exactly the source record (feedback item 5): compare with the hash taken at extraction time,
# and check that every variant is an exact substring of original_text at the recorded position.
assert (D21["original_text"].map(sha1).values == df["original_sha1"].values).all(), "original_text was modified!"
assert all(o[s:e] == v for o, s, e, v in zip(df["original_text"], df["char_start"], df["char_end"], df["variant"]))
print("original_text untouched and variants taken from it ✔")

## Step 5. Manual check of 50 rows
The sheet holds 50 rows spread over the three sources, including some rows flagged as quotes so that quote detection is checked too.
**What you do:** open `manual/D2_check50_sheet.csv`. For each row and each field, if the automatic value is **wrong**, write the correct value in the matching `correct_…` column; leave it empty when the automatic value is right. Put `y` in `reviewed`. Save as `manual/D2_check50_filled.csv`, upload, rerun.
If the same error repeats, fix the rule in Part B, run all again, and report both error counts.

In [ ]:
# Reuse the reviewed examples only when occurrence and context still match.
checked = pd.DataFrame()
check_path = MANUAL / "D2_check50_filled.csv"
if check_path.exists():
    checked = match_sheet(read_csv(check_path), df)
    checked = checked[checked["reviewed"].str.strip().str.lower().isin(["y", "yes", "1", "x"])]
    write_csv(checked, REP2 / "matched_prior_check50.csv")
CHECK_COMPLETE = len(checked) == 50 and checked["source"].nunique() == 3
if len(checked):
    res = []
    before = AUTO_BEFORE_REVIEW.set_index("id")
    after = df.set_index("id")
    for field in CHECK_FIELDS:
        gold = checked["correct_" + field].where(checked["correct_" + field].str.strip().ne(""), checked["auto_" + field])
        auto = before.loc[checked["id"], field].reset_index(drop=True)
        final = after.loc[checked["id"], field].reset_index(drop=True)
        wrong = auto.ne(gold.reset_index(drop=True))
        res.append(dict(field=field, rows_checked=len(checked), wrong=int(wrong.sum()),
                        error_rate=round(wrong.mean(), 3), remaining_after_corrections=int(final.ne(gold.reset_index(drop=True)).sum())))
    errors = pd.DataFrame(res)
    write_csv(errors, REP2 / "table_check50_errors.csv")
    print(errors.to_string(index=False))
    CHECK_COMPLETE = CHECK_COMPLETE and errors["remaining_after_corrections"].eq(0).all()
elif (REP2 / "table_check50_errors.csv").exists():
    (REP2 / "table_check50_errors.csv").unlink()
# New rules/context need an updated sheet when prior gold values no longer agree.
parts = []
for source, count in {"sfc": 17, "gulf": 17, "sada": 16}.items():
    pool = df[df["source"] == source]
    quotes = pool[pool["quote_flag"] == "true"]
    nq = min(4, len(quotes), count)
    parts += [quotes.sample(nq, random_state=SEED), pool[pool["quote_flag"] == "false"].sample(count - nq, random_state=SEED)]
check = pd.concat(parts).sample(frac=1, random_state=SEED)
sheet = check[["id", "source", "occurrence_key", "host_sentence", "prev_context", "original_text"]].copy()
sheet["reviewed"] = ""
for field in CHECK_FIELDS:
    sheet["auto_" + field] = check[field].values
    sheet["correct_" + field] = ""
write_csv(sheet, MANUAL / "D2_check50_sheet.csv")
print("50-row validation:", "complete" if CHECK_COMPLETE else "needs review; fill the new D2_check50_sheet.csv and save D2_check50_filled.csv")

## Step 6 — natural proportions and better rare-label evidence
The 100-row check is source-stratified 30/35/35. Per-source shares estimate natural use within each source; the combined share refers to that chosen mix, not the full corpus's source proportions.

The earlier quick labels are matched by occurrence and context. If fewer than 100 survive, fill the new `D2_rare100_sheet.csv` and save `D2_rare100_filled.csv`.

`D2_rate_survey_sheet.csv` samples up to 20 rows per source × stratum. Fill quick_label and save `D2_rate_survey_filled.csv` for stronger sampling estimates. A separate `D2_rare_candidates_sheet.csv` helps locate actual SKEPTIC/DEFER examples; fill quick_label and save `D2_rare_candidates_filled.csv`. These labels guide sampling only; final Phase 3 label columns remain empty.

In [ ]:
def assign_stratum(r):
    if has_skeptic_cue(r["host_sentence"], r["prev_context"]):
        return "skeptic_cue"
    if r["position"] == "alone" or r["n_words"] <= 5:
        return "short_or_alone"
    if answers_question(r["prev_context"]):
        return "answers_question"
    if r["second_person"] == "true":
        return "addressed"
    if r["verb_cue"] == "me_past" or has_yakoon(r["host_sentence"]):
        return "past_or_yakoon"
    return "other"

df["stratum_name"] = df.apply(assign_stratum, axis=1)
eligible = df[df["quote_flag"] == "false"].copy()
SHEET_COLUMNS = ["id", "source", "occurrence_key", "host_sentence", "prev_context"]
def blank_label_sheet(frame, filename):
    out = frame[SHEET_COLUMNS].copy()
    out["quick_label"] = ""
    write_csv(out, MANUAL / filename)

QUICK = pd.DataFrame()
matched_random = pd.DataFrame()
for filename, kind in [("D2_rare100_filled.csv", "random100"),
                       ("D2_stratum_extra_filled.csv", "stratum_extra"),
                       ("D2_rate_survey_filled.csv", "rate_survey")]:
    path = MANUAL / filename
    if not path.exists():
        continue
    q = match_sheet(read_csv(path), eligible)
    q["quick_label"] = q["quick_label"].str.strip().str.upper()
    q = q[q["quick_label"].isin(ALL_QUICK_LABELS)]
    q = q.merge(eligible[["id", "stratum_name"]], on="id", how="left", validate="one_to_one")
    q["set"] = kind
    if kind == "random100":
        matched_random = q.copy()
    QUICK = pd.concat([QUICK, q], ignore_index=True)
if len(QUICK):
    QUICK = QUICK.drop_duplicates("occurrence_key", keep="last")
RANDOM_COMPLETE = len(matched_random) == 100 and matched_random["source"].value_counts().to_dict() == RARE_CHECK_SIZES
random_draw = pd.concat([eligible[eligible["source"] == s].sample(n, random_state=SEED) for s, n in RARE_CHECK_SIZES.items()]).sample(frac=1, random_state=SEED)
blank_label_sheet(random_draw, "D2_rare100_sheet.csv")
if RANDOM_COMPLETE:
    tab = []
    for label in ALL_QUICK_LABELS:
        k = int(matched_random["quick_label"].eq(label).sum()); lo, hi = wilson(k, 100)
        tab.append(dict(label=label, count_in_100=k, estimate_in_1000=k * 10,
                        enough="Yes" if k >= 5 else "No; add more", source_mix_share=k / 100, ci95=f"{lo:.1%}–{hi:.1%}"))
    write_csv(pd.DataFrame(tab), REP2 / "table_rare100.csv")
    by_source = pd.crosstab(matched_random["quick_label"], matched_random["source"], normalize="columns").reindex(ALL_QUICK_LABELS).fillna(0)
    write_csv(by_source.reset_index(), REP2 / "table_rq1_natural_proportions_by_source.csv")
else:
    print(f"Random check: {len(matched_random)} of 100 prior rows matched. Complete the fresh random sheet before final export.")
    for name in ["table_rare100.csv", "table_rq1_natural_proportions_by_source.csv"]:
        path = REP2 / name
        if path.exists():
            path.unlink()

used = set(QUICK["occurrence_key"]) if len(QUICK) else set()
survey = []
for (source, stratum), group in eligible.groupby(["source", "stratum_name"], sort=True):
    group = group[~group["occurrence_key"].isin(used)]
    survey.append(group.sample(min(RATE_SURVEY_PER_CELL, len(group)), random_state=SEED))
blank_label_sheet(pd.concat(survey).sample(frac=1, random_state=SEED), "D2_rate_survey_sheet.csv")

# Human-verified rare candidates become explicit strata; never infer their function from cue words.
verified = pd.DataFrame()
path = MANUAL / "D2_rare_candidates_filled.csv"
if path.exists():
    verified = match_sheet(read_csv(path), eligible)
    verified["quick_label"] = verified["quick_label"].str.strip().str.upper()
    verified = verified[verified["quick_label"].isin(SIX_LABELS)]
    for label in SIX_LABELS:
        ids = verified.loc[verified["quick_label"] == label, "id"]
        eligible.loc[eligible["id"].isin(ids), "stratum_name"] = "verified_" + label.lower()
candidate_parts = []
verified_ids = set(verified["id"]) if len(verified) else set()
for source, group in eligible.groupby("source", sort=True):
    group = group[~group["id"].isin(verified_ids)].copy()
    group["candidate_score"] = [int(has_skeptic_cue(h, p)) + int(answers_question(p)) + int(n <= 8)
                                for h, p, n in zip(group["host_sentence"], group["prev_context"], group["n_words"])]
    group = group[group["candidate_score"] > 0].sample(frac=1, random_state=SEED).sort_values("candidate_score", ascending=False, kind="stable")
    candidate_parts.append(group.head(RARE_CANDIDATES_PER_SOURCE))
blank_label_sheet(pd.concat(candidate_parts), "D2_rare_candidates_sheet.csv")
print("Matched quick labels:", len(QUICK), "| separately verified candidates:", len(verified))

## Step 7 — choose source quotas using the reviewed evidence
The allocator keeps 300/350/350, retains at least one row from each populated stratum, and tries for 60 expected examples per main label, then 50. It uses source-specific reviewed rates when at least 10 reviews exist, otherwise a clearly marked pooled-stratum estimate. Sparse cells have zero planning evidence until reviewed. Verified candidate strata use their human quick labels. Estimates remain planning estimates, not formal gold labels.

If no feasible allocation meets 50, a provisional sample and actionable sheets are exported. The notebook does not quietly label an inadequate sample final. The allocator uses [SciPy's integer optimization API](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.milp.html).

In [ ]:
for source, count in SOURCE_TOTALS.items():
    have = int(eligible["source"].eq(source).sum())
    assert have >= count, f"{source}: only {have} eligible rows for quota {count}"

cells, rate_rows = [], []
for (source, stratum), group in eligible.groupby(["source", "stratum_name"], sort=True):
    local = QUICK[(QUICK["source"] == source) & (QUICK["stratum_name"] == stratum)] if len(QUICK) else pd.DataFrame()
    pooled = QUICK[QUICK["stratum_name"] == stratum] if len(QUICK) else pd.DataFrame()
    if stratum.startswith("verified_"):
        label = stratum[len("verified_"):].upper()
        rate = {lab: float(lab == label) for lab in SIX_LABELS}
        evidence = "human-verified candidate labels"; count = len(group)
    else:
        evidence_rows = local if len(local) >= MIN_RATE_REVIEWS else pooled
        evidence = "source-by-stratum" if len(local) >= MIN_RATE_REVIEWS else "pooled-stratum (assumption)"
        count = len(evidence_rows)
        rate = evidence_rows["quick_label"].value_counts(normalize=True).to_dict() if count >= MIN_RATE_REVIEWS else {}
        if count < MIN_RATE_REVIEWS:
            evidence = "insufficient reviews; planning rates set to zero"
    cells.append(dict(source=source, stratum=stratum, available=len(group), reviews=count, evidence=evidence))
    rate_rows.append([rate.get(label, 0.0) for label in SIX_LABELS])
alloc_df = pd.DataFrame(cells)
rates = np.asarray(rate_rows)
for i, label in enumerate(SIX_LABELS):
    alloc_df["rate_" + label] = rates[:, i]
allocation = plan_quotas(alloc_df["source"], alloc_df["available"], rates, SOURCE_TOTALS, target=PREFERRED_EXPECTED)
if allocation is None:
    allocation = plan_quotas(alloc_df["source"], alloc_df["available"], rates, SOURCE_TOTALS, target=MIN_EXPECTED)
RARE_TARGET_MET = allocation is not None
if allocation is None:
    # Diagnostics only: preserve source totals and give each nonempty cell support.
    allocation = np.zeros(len(alloc_df), dtype=int)
    for source, total in SOURCE_TOTALS.items():
        idx = alloc_df.index[alloc_df["source"] == source].tolist()
        for i in idx:
            allocation[i] = 1
        for _ in range(total - len(idx)):
            choices = [i for i in idx if allocation[i] < alloc_df.at[i, "available"]]
            i = max(choices, key=lambda j: alloc_df.at[j, "available"] / (allocation[j] + 1))
            allocation[i] += 1
    print("No feasible 50-per-label allocation yet. Fill rate survey and/or verify more rare candidates; output is provisional.")
alloc_df["drawn"] = allocation
alloc_df["share_of_source"] = alloc_df["drawn"] / alloc_df["source"].map(SOURCE_TOTALS)
write_csv(alloc_df, REP2 / "table_strata_allocation.csv")
expected = rates.T @ allocation
exp_df = pd.DataFrame({"label": SIX_LABELS, "expected_in_D2_2": expected.round(2), "meets_50": expected >= MIN_EXPECTED - 1e-7})
write_csv(exp_df, REP2 / "table_expected_label_counts.csv")
print(exp_df.to_string(index=False))
rng = np.random.default_rng(SEED)
picked = []
for row in alloc_df.itertuples():
    group = eligible[(eligible["source"] == row.source) & (eligible["stratum_name"] == row.stratum)]
    picked.append(group.sample(row.drawn, random_state=int(rng.integers(1e9))))
sample = pd.concat(picked).sample(frac=1, random_state=SEED)
D22 = sample.copy()
D22["stratum"] = D22["source"] + "_" + D22["stratum_name"]
D22["label"], D22["context_needed"] = "", ""
D22 = D22[D22_COLUMNS]
RUN_STATUS = "READY_FOR_REPORT_REVIEW" if RARE_TARGET_MET and CHECK_COMPLETE and RANDOM_COMPLETE and not D21_MAX_ROWS else "NEEDS_REVIEW"
final_path = PROCESSED / "D2.2_annotation_sample.csv"
if RUN_STATUS == "READY_FOR_REPORT_REVIEW":
    write_csv(D22, final_path)
    provisional = PROCESSED / "D2.2_PROVISIONAL_annotation_sample.csv"
    if provisional.exists():
        provisional.unlink()
else:
    if final_path.exists():
        final_path.rename(PROCESSED / ("D2.2_previous_not_current_" + dt.datetime.now().strftime("%Y%m%d_%H%M%S") + ".csv"))
    write_csv(D22, PROCESSED / "D2.2_PROVISIONAL_annotation_sample.csv")
w = alloc_df[["source", "stratum", "available", "drawn"]].rename(columns={"available": "population"}).copy()
w["weight"] = w["population"] / w["drawn"]
w["stratum"] = w["source"] + "_" + w["stratum"]
write_csv(w[["stratum", "source", "population", "drawn", "weight"]], PROCESSED / "D2.2_sampling_weights.csv")
print("RUN STATUS:", RUN_STATUS)

## Automatic checks (the final submission checklist)

In [ ]:
d21 = read_csv(PROCESSED / "D2.1_master_collection.csv")
d22 = D22.copy()
checks = {
    "D2.1 has exactly the 11 column names, in order": list(d21.columns) == D21_COLUMNS,
    "D2.1 ids are unique": d21["id"].is_unique,
    "D2.1 source values are sfc/gulf/sada": d21["source"].isin(["sfc", "gulf", "sada"]).all(),
    "D2.1 position values allowed": d21["position"].isin(["first", "middle", "last", "alone"]).all(),
    "D2.1 verb_cue values allowed": d21["verb_cue"].isin(["me_future", "me_past", "other", "none"]).all(),
    "D2.1 boolean columns are true/false": all(d21[c].isin(["true", "false"]).all() for c in ["future_marker", "second_person", "quote_flag"]),
    "Every host sentence has 4–80 words": d21["host_sentence"].map(n_words).between(MIN_WORDS, MAX_WORDS).all(),
    "Every variant appears in its original_text": all(v in o for v, o in zip(d21["variant"], d21["original_text"])),
    "D2.1 has about 3,000 rows or more": len(d21) >= 3000,
    "D2.2 has the 11 columns + stratum, label, context_needed": list(d22.columns) == D22_COLUMNS,
    "D2.2 has exactly 1,000 rows": len(d22) == 1000,
    "D2.2 is 300 sfc / 350 gulf / 350 sada": d22["source"].value_counts().to_dict() == SOURCE_TOTALS,
    "D2.2 has no quote_flag = true": (d22["quote_flag"] == "false").all(),
    "D2.2 stratum filled; label and context_needed empty": (d22["stratum"] != "").all() and (d22["label"] == "").all() and (d22["context_needed"] == "").all(),
    "D2.2 ids are unique and all exist in D2.1": d22["id"].is_unique and d22["id"].isin(d21["id"]).all(),
    "D2.2 rows are identical to their D2.1 rows": (d22[D21_COLUMNS].merge(d21, on=D21_COLUMNS, how="left", indicator=True)["_merge"] == "both").all(),
    "D2.2 is shuffled (not sorted by source or id)": not d22["id"].is_monotonic_increasing and d22["source"].ne(d22["source"].shift()).sum() > 100,
}
checks["No URL/user fragments in working copies"] = not d21["host_sentence"].str.contains(r"https?://|www\.|@\w+", regex=True).any()
for k, v in checks.items():
    print(("✔ " if v else "✘ ") + k)
STRUCTURE_OK = bool(all(checks.values()))
if not STRUCTURE_OK:
    RUN_STATUS = "NEEDS_REVIEW"
readiness = dict(structural_checks=STRUCTURE_OK, manual_check50_complete=bool(CHECK_COMPLETE),
                 random100_complete=bool(RANDOM_COMPLETE), all_six_expected_ge50=bool(RARE_TARGET_MET),
                 full_master_not_capped=not bool(D21_MAX_ROWS), report_review_pending=True)
(REP2 / "submission_readiness.json").write_text(json.dumps({"status": RUN_STATUS, "checks": readiness}, indent=2), encoding="utf-8")
if not STRUCTURE_OK and final_path.exists():
    final_path.replace(PROCESSED / "D2.2_PROVISIONAL_annotation_sample.csv")
print("Status:", RUN_STATUS, "| report writing, pagination and linguistic example review remain separate.")

## Step 8. Numbers for the D2.3 report
Everything below is computed from **D2.1** (the natural collection), except the strata table, which describes D2.2.
All tables are saved in `reports/phase2/` and summarised in `reports/phase2/phase2_numbers.md`.

In [ ]:
T = {}
T["variants"] = d21.groupby(["source", "variant"]).size().rename("count").reset_index().sort_values(["source", "count"], ascending=[True, False])
T["variants_total"] = d21["variant"].value_counts().rename_axis("variant").reset_index(name="count")
for col in ["position", "verb_cue", "future_marker", "second_person", "quote_flag"]:
    T[col] = pd.crosstab(d21[col], d21["source"], margins=True, margins_name="total").reset_index()
lens = d21["host_sentence"].map(n_words)
T["length"] = lens.groupby(d21["source"]).describe()[["count", "mean", "50%", "min", "max"]].round(1).rename(columns={"50%": "median"}).reset_index()
T["prev_context"] = (d21["prev_context"] != "").groupby(d21["source"]).agg(["sum", "mean"]).rename(columns={"sum": "rows_with_prev", "mean": "share"}).round(3).reset_index()
T["d22_strata"] = d22["stratum"].value_counts().rename_axis("stratum").reset_index(name="rows")
if "dialect" in df.columns:
    T["sada_dialects"] = df[df["source"] == "sada"]["dialect"].value_counts().rename_axis("speaker_dialect").reset_index(name="rows")
for name, t in T.items():
    write_csv(t, REP2 / f"table_{name}.csv")

md_lines = ["# Phase 2 numbers for the D2.3 report", "", f"Generated {dt.date.today()} with seed {SEED}.", ""]
def md_table(t):
    t = t.astype(str)
    return ["| " + " | ".join(t.columns) + " |", "|" + "---|" * len(t.columns)] + ["| " + " | ".join(r) + " |" for r in t.values]
for title, path in [("Rows per stage", "table_counts_by_stage.csv"), ("Spelling variants (all sources)", "table_variants_total.csv"),
                    ("Position", "table_position.csv"), ("Verb cue", "table_verb_cue.csv"), ("Future marker", "table_future_marker.csv"),
                    ("Second person", "table_second_person.csv"), ("Quote flag", "table_quote_flag.csv"), ("Host sentence length (words)", "table_length.csv"),
                    ("Rows with prev_context", "table_prev_context.csv"), ("100-row rare-meaning check", "table_rare100.csv"),
                    ("Natural proportions by source (RQ1)", "table_rq1_natural_proportions_by_source.csv"),
                    ("Expected label counts in D2.2", "table_expected_label_counts.csv"), ("50-row manual check: errors", "table_check50_errors.csv"),
                    ("Strata allocation in D2.2", "table_strata_allocation.csv")]:
    p = REP2 / path
    md_lines += [f"## {title}", ""] + (md_table(read_csv(p)) if p.exists() else ["_not available yet (manual step pending)_"]) + [""]
(REP2 / "phase2_numbers.md").write_text("\n".join(md_lines), encoding="utf-8")
print("wrote reports/phase2/phase2_numbers.md")
print(T["variants_total"].head(15).to_string(index=False))
manifest = {"revision": REVISION, "seed": SEED, "status": RUN_STATUS, "master_caps": D21_MAX_ROWS,
            "reviewer": REVIEWER_NAME, "input_revisions": ACCESS,
            "packages": {p: md.version(p) for p in ["camel-tools", "numpy", "pandas", "datasketch", "huggingface_hub", "scipy"]},
            "input_sha256": {str(p.relative_to(RAW)): sha256_file(p)
                             for p in [*[sfc_dir / f for f in sfc_files], *sada_paths, gulf_dir / gulf_csv_name, tanzil_path]},
            "output_sha256": {p.name: sha256_file(p) for p in PROCESSED.glob("*.csv")}}
(REP2 / "run_manifest.json").write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")
draft = ["# D2.3 report draft from the run", "", "Status: " + RUN_STATUS, "",
         "Previous feedback is addressed through preserved original text, boundary rules, OTHER, independent random-check proportions and explicit rare-label readiness.", "",
         "## Validation", f"Prior manual validation was confirmed by {REVIEWER_NAME}. Matched sheets and correction logs are included. Changed context is not silently reused.", "",
         "## Limitations", "Quick labels estimate sampling yields; they are not gold annotations. Pooled-stratum rates assume similar label proportions across sources and are identified in the allocation table. Mixed dialects and multi-speaker SADA segments need explicit discussion. Bare formulas are excluded by the 4-word minimum.", "",
         "## Required writing before submission", "Use phase2_numbers.md and table_full_cleaned_variants.csv to write a 4–6 page report. Explain source licences, pipeline, filtering, counts, metadata error patterns, sampling shares, the 100-row check and biases. Review Phase 1 examples for WISH/CLOSE and HEDGE/WISH boundaries. No literature report is claimed to be generated here.", ""]
(REP2 / "D2.3_report_draft.md").write_text("\n".join(draft), encoding="utf-8")
next_steps = ["# Next steps", "", "Run status: " + RUN_STATUS, ""]
if not CHECK_COMPLETE:
    next_steps.append("- Review D2_check50_sheet.csv and save D2_check50_filled.csv with reviewed=y and corrections as needed.")
if not RANDOM_COMPLETE:
    next_steps.append("- Complete all 100 quick labels in D2_rare100_sheet.csv; save D2_rare100_filled.csv.")
if not RARE_TARGET_MET:
    next_steps.append("- Complete D2_rate_survey_sheet.csv and/or D2_rare_candidates_sheet.csv; save the corresponding _filled.csv files. Verify real meanings; do not label cue words automatically.")
if D21_MAX_ROWS:
    next_steps.append("- Remove trial master caps and rerun the cleaning pipeline for the full collection.")
next_steps += ["- After new reviews, rerun Step 4 onward (Step 4 applies corrections before strata and sampling).",
               "- Download the output ZIP and the executed notebook using File → Download → Download .ipynb; return both for review.",
               "- Finish and visually check the separate D2.3 report before submission."]
(REP2 / "NEXT_STEPS.md").write_text("\n".join(next_steps), encoding="utf-8")

## Download the outputs
The ZIP includes the master collection, final or explicitly provisional sample, metadata corrections, review sheets, full spelling counts, expected-label estimates, run manifest, readiness status and next steps. It does not include raw corpora.

Also download this executed notebook from File → Download → Download .ipynb. Return both files. If the status is NEEDS_REVIEW, use NEXT_STEPS.md to identify which sheets need work. Keep the separate D2.3 report under review until new counts are known.

In [ ]:
import zipfile
zip_path = PROJECT_DIR / "inshallah_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:   # raw data is NOT included (licences, size)
    for folder in [PROCESSED, REP1, REP2, MANUAL]:
        for f in folder.rglob("*"):
            if f.is_file():
                z.write(f, f.relative_to(PROJECT_DIR))
print("Created", zip_path, f"({zip_path.stat().st_size/1e6:.1f} MB)")
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))